In [1]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, PDFSearchTool
import litellm
litellm.ssl_verify = False

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from crewai_tools import PDFSearchTool
import inspect

print(inspect.signature(PDFSearchTool))

(*, name: str = "Search a PDF's content", description: str = "A tool that can be used to semantic search a query from a PDF's content.", env_vars: list[crewai.tools.base_tool.EnvVar] = <factory>, args_schema: type[pydantic.main.BaseModel] = <class 'crewai_tools.tools.pdf_search_tool.pdf_search_tool.PDFSearchToolSchema'>, description_updated: bool = False, cache_function: Annotated[collections.abc.Callable[..., Any], BeforeValidator(func=<function string_to_callable at 0x000001D511411EE0>, json_schema_input_type=PydanticUndefined), PlainSerializer(func=<function callable_to_string at 0x000001D511412020>, return_type=<class 'str'>, when_used='json'), WithJsonSchema(json_schema={'type': 'string'}, mode=None)] = <function _default_cache_function at 0x000001D511411D00>, result_as_answer: bool = False, max_usage_count: int | None = None, current_usage_count: int = 0, summarize: bool = False, similarity_threshold: float = 0.6, limit: int = 5, collection_name: str = 'rag_tool_collection', adap

In [4]:
llm = LLM(model="gemini/gemini-2.5-flash")

In [5]:
web_search_tool = SerperDevTool()

Creating pdf search tool

In [6]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# import warnings
# warnings.filterwarnings('ignore') #Keeps Jupyter Notebook clean (not part of functionality)

# pdf_search_tool = PDFSearchTool(
#     pdf=r"E:\MYSELF\CODE\AIAI\LLM\GIT_ME\llm\crewai_agent_tool_task_tool\The_Daily_Dish_FAQ.pdf",
#     config={
#         "embedding_model": {
#             "provider": "huggingface",  # or: "google-generativeai", "cohere", "ollama", ...
#             "config": {
#                 # Model identifier for the chosen provider. "model" will be auto-mapped to "model_name" internally.
#                 "model": "sentence-transformers/all-MiniLM-L6-v2",
                
#             },
#         }
#     }
# )

In [13]:
import os
from crewai.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Định nghĩa đường dẫn file PDF và thư mục lưu DB cục bộ
PDF_PATH = r"E:\MYSELF\CODE\AIAI\LLM\GIT_ME\llm\crewai_agent_tool_task_tool\The_Daily_Dish_FAQ.pdf"
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DB_DIR = os.path.join(CURRENT_DIR, "langchain_chroma_db")

langchain_embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 3. Tiến hành nạp PDF và Vector hóa (Chỉ thực hiện nếu DB chưa tồn tại để tiết kiệm thời gian)
if not os.path.exists(DB_DIR):
    print("Đang nạp file PDF và khởi tạo Vector Database...")
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()
    
    # Chia nhỏ văn bản thành các đoạn thích hợp để AI dễ tìm kiếm
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)
    
    # Lưu vào Chroma DB sử dụng bộ nhúng LangChain HuggingFace
    vector_store = Chroma.from_documents(
        documents=splits, 
        embedding=langchain_embedder, 
        persist_directory=DB_DIR
    )
else:
    # Nếu đã tạo rồi thì chỉ cần load lại database lên sử dụng
    vector_store = Chroma(
        persist_directory=DB_DIR, 
        embedding_function=langchain_embedder
    )

@tool("Tìm kiếm trong tài liệu FAQ")
def pdf_search_tool(query: str) -> str:
    """Hữu ích khi bạn cần tìm kiếm thông tin, quy định, hoặc câu trả lời 
    liên quan đến The Daily Dish từ file tài liệu FAQ."""
    
    # Tìm kiếm top 3 đoạn văn bản có độ tương đồng cao nhất với câu hỏi
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    relevant_docs = retriever.invoke(query)
    
    # Kết hợp các kết quả tìm được thành chuỗi văn bản trả về cho Agent
    results = "\n\n".join([doc.page_content for doc in relevant_docs])
    return results if results else "Không tìm thấy thông tin phù hợp trong tài liệu."

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2797.22it/s]
Ignoring wrong pointing object 89 0 (offset 0)


Đang nạp file PDF và khởi tạo Vector Database...


In [14]:
agent_centric_agent = Agent(
    role="The Daily Dish Inquiry Specialist",
    goal="""Accurately answer customer questions about The Daily Dish restaurant. 
    You must decide whether to use the restaurant's FAQ PDF or a web search to find the best answer.""",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You have access to two tools: one for searching the restaurant's FAQ document and another for searching the web.
    Your job is to analyze the user's question and choose the most appropriate tool to find the information needed to provide a helpful response.""",
    tools=[pdf_search_tool, web_search_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [15]:
agent_centric_task = Task(
    description="Answer the following customer query: '{customer_query}'. "
                "Analyze the question and use the tools at your disposal (PDF search or web search) to find the most relevant information. "
                "Synthesize the findings into a clear and friendly response.",
    expected_output="A comprehensive and well-formatted answer to the customer's query.",
    agent=agent_centric_agent
)

In [16]:
agent_centric_crew = Crew(
    agents=[agent_centric_agent],
    tasks=[agent_centric_task],
    process=Process.sequential,
    verbose=False
)

In [18]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_agent_centric = await agent_centric_crew.kickoff_async(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_agent_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")


Welcome to The Daily Dish Chatbot!
What would you like to know? (Type 'exit' to quit)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│  Task: Answer the following customer query: 'what is the phone number?'. Analyze the question and use the       │
│  tools at your disposal (PDF search or web search) to find the most relevant information. Synthesize the        │
│  findings into a clear and friendly response.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool tim_kiem_trong_tai_lieu_faq executed with result: The  Daily  Dish  -  Frequently  Asked  Questions   General  Information  &  Location   1.   Q:  What  are  your  hours  of  operation?      A:  The  Daily  Dish  is  open  Monday  to  Friday  from  1...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  You can reach The Daily Dish at (555) 123-4567.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- The Daily Dish Assistant ---
You can reach The Daily Dish at (555) 123-4567.
--------------------------------


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The name of the restaurant is The Daily Dish.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- The Daily Dish Assistant ---
The name of the restaurant is The Daily Dish.
--------------------------------


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thank you for chatting. Have a great day!


ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30)


Task centric

In [19]:
task_centric_agent = Agent(
    role="Customer Service Specialist",
    goal="Provide exceptional customer service by following a multi-step process to answer customer questions accurately.",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You are an expert at following instructions. You will be given a sequence of tasks to complete.
    For each task, you will be provided with the specific tool needed to accomplish it.
    Your job is to execute each task diligently and pass the results to the next step.""",
    tools=[], # The agent is not given any tools directly
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [20]:
faq_search_task = Task(
    description="Search the restaurant's FAQ PDF for information related to the customer's query: '{customer_query}'.",
    expected_output="A snippet of the most relevant information from the PDF, or a statement that the information was not found.",
    tools=[pdf_search_tool], # Tool assigned directly to the task
    agent=task_centric_agent
)

response_drafting_task = Task(
    description="Using the information gathered from the FAQ search, draft a friendly and comprehensive response to the customer's query: '{customer_query}'.",
    expected_output="The final, customer-facing response.",
    agent=task_centric_agent,
    context=[faq_search_task]
)

In [21]:
task_centric_crew = Crew(
    agents=[task_centric_agent],
    tasks=[faq_search_task, response_drafting_task],
    process=Process.sequential,
    verbose=True
)

In [22]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_task_centric = await task_centric_crew.kickoff_async(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_task_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")


Welcome to The Daily Dish Chatbot!
What would you like to know? (Type 'exit' to quit)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f35f0cda-3a39-49f3-82da-5d9c29b89c20                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the restaurant's FAQ PDF for information related to the customer's query: 'what is the daily      │
│  dish'.                                                                                                         │
│  ID: ea19061b-2ae8-4635-a3f0-5a9a002f85ec                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│  Task: Search the restaurant's FAQ PDF for information related to the customer's query: 'what is the daily      │
│  dish'.                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tim_kiem_trong_tai_lieu_faq                                                                              │
│  Args: {'query': 'what is the daily dish'}                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: f35f0cda-3a39-49f3-82da-5d9c29b89c20                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

An error occurred: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


Thank you for chatting. Have a great day!
